In [1]:
# Core data manipulation
import pandas as pd
import numpy as np
from numpy import polyfit, poly1d

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import plotly.express as px

from vega_datasets import data

In [2]:
owid = pd.read_csv('owid-covid-data.csv')
display(owid.head())

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [3]:
owid['stringency_index']

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
429430    NaN
429431    NaN
429432    NaN
429433    NaN
429434    NaN
Name: stringency_index, Length: 429435, dtype: float64

### Visualization #1

In [4]:
import pycountry
from vega_datasets import data as vega_data

deaths = (
    owid.groupby(['iso_code', 'location'])['total_deaths_per_million']
    .max()
    .reset_index()
)

# Convert ISO alpha-3 to numeric ID to match the map
def to_numeric(iso3):
    try:
        return int(pycountry.countries.get(alpha_3=iso3).numeric)
    except:
        return None

deaths['id'] = deaths['iso_code'].apply(to_numeric)
deaths = deaths.dropna(subset=['id'])
deaths['id'] = deaths['id'].astype(int)

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

alt.Chart(world).mark_geoshape().encode(
    color=alt.Color(
        'total_deaths_per_million:Q',
        scale=alt.Scale(scheme='yelloworangered',reverse=False),
        title='Deaths per Million'),
    tooltip=['location:N', 'total_deaths_per_million:Q']).transform_lookup(
    lookup='id',
    from_=alt.LookupData(
        data=deaths,
        key='id',
        fields=['total_deaths_per_million', 'location']
    )).project('naturalEarth1').properties(
    width=800,
    height=450,
    title='COVID-19 Deaths per Million by Country'
)

alt.Chart(...)

### Visualization #2

In [5]:
alt.data_transformers.disable_max_rows()

# Filter to 2020-2022, monthly average
owid['date'] = pd.to_datetime(owid['date'])
stringency_monthly = (
    owid[
        (owid['date'] >= '2020-01-01') &
        (owid['date'] <= '2022-12-31') &
        owid['stringency_index'].notna()
    ]
    .groupby(['location', pd.Grouper(key='date', freq='MS')])['stringency_index']
    .mean()
    .reset_index()
)

highlight = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']

# --- Background grey lines (all other countries) ---
grey = alt.Chart(
    stringency_monthly[~stringency_monthly['location'].isin(highlight)]).mark_line(opacity=0.08, color='grey', strokeWidth=0.8).encode(
    x=alt.X('date:T', title='', axis=alt.Axis(format='%b %Y', tickCount={'interval': 'month', 'step': 3})),
    y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
    detail='location:N'
)

# --- Highlighted country lines ---
color_scale = alt.Scale(
    domain=highlight,
    range=['#1f77b4', '#d62728', '#2ca02c', '#aaaaaa', '#ff7f0e', '#9467bd']
    # US=blue, China=red, NZ=green, Sweden=grey, Brazil=orange, Germany=purple
)

lines = alt.Chart(
    stringency_monthly[stringency_monthly['location'].isin(highlight)]).mark_line(strokeWidth=2.2).encode(
    x='date:T',
    y='stringency_index:Q',
    color=alt.Color('location:N', scale=color_scale, title='Country'),
    tooltip=[
        alt.Tooltip('location:N', title='Country'),
        alt.Tooltip('date:T', title='Date', format='%b %Y'),
        alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')
    ]
)

# --- Key event annotations ---
events = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',       'y': 96},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',          'y': 88},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',           'y': 80},
    {'date': '2020-06-08', 'label': 'NZ Reopens',            'y': 96},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',       'y': 88},
    {'date': '2021-07-01', 'label': 'Delta Wave',             'y': 80},
    {'date': '2021-12-01', 'label': 'Omicron Wave',          'y': 96},
    {'date': '2022-03-01', 'label': 'Most Restrictions Lift','y': 88},
])
events['date'] = pd.to_datetime(events['date'])

annotation_rules = alt.Chart(events).mark_rule(
    strokeDash=[4, 4], opacity=0.4, color='black', strokeWidth=1).encode(x='date:T')

annotation_text = alt.Chart(events).mark_text(
    align='left', dx=4, fontSize=9, color='#333333', fontStyle='italic').encode(
    x='date:T',
    y=alt.Y('y:Q'),
    text='label:N')

# --- Combine ---
chart = (grey + lines + annotation_rules + annotation_text).properties(
    width=850,
    height=480,
    title=alt.TitleParams(
        'COVID-19 Oxford Stringency Index by Country (2020–2022)',
        fontSize=15
    )).configure_axis(
    grid=True, gridOpacity=0.2).configure_view(
    strokeWidth=0
)

chart

alt.LayerChart(...)

### Visualization #3

In [6]:
owid = pd.read_csv('owid-covid-data.csv')

In [7]:
# setting date to date time format
owid['date'] = pd.to_datetime(owid['date'])

# First confirmed death per country 
first_death = (
    owid[owid['total_deaths'] > 0]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'first_death_date'})
)

# First strong restriction date (stringency ≥ 70) 
strong_restriction = (
    owid[owid['stringency_index'] >= 70]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'restriction_date'})
)

# Merge
country_dates = pd.merge(first_death, strong_restriction, on='location', how='inner')

# Calculate days between
country_dates['days_to_restrictions'] = (
    country_dates['restriction_date'] - country_dates['first_death_date']
).dt.days

# Get deaths per million 
deaths_pm = (
    owid.groupby('location')['total_deaths_per_million']
    .max()
    .reset_index()
)

# Get continent + population
meta = (
    owid.groupby('location')
    .agg({
        'continent': 'first',
        'population': 'first'
    })
    .reset_index()
)

# Final dataset 
country_owid = (
    country_dates
    .merge(deaths_pm, on='location')
    .merge(meta, on='location')
)

# Drop missing continent rows 
country_owid = country_owid[country_owid['continent'].notna()]

country_owid.head()

,location,first_death_date,restriction_date,days_to_restrictions,total_deaths_per_million,continent,population
0,Afghanistan,2020-03-29,2020-04-05,7,197.10,Asia,41128772
1,Albania,2020-03-15,2020-03-13,-2,1274.93,Europe,2842318
2,Algeria,2020-03-29,2020-03-23,-6,151.31,Africa,44903228
3,Angola,2020-05-24,2020-03-27,-58,54.36,Africa,35588996
4,Argentina,2020-03-08,2020-03-19,11,2877.54,South America,45510324


In [8]:
# countries to label
label_countries = [
    "United States", "Italy", "China", "India", "Brazil",
    "United Kingdom", "Sweden", "New Zealand"
]

# Base scatterplot
points = (
    alt.Chart(country_owid)
    .mark_circle(opacity=0.7, stroke='black', strokeWidth=0.3)
    .encode(
        x=alt.X(
            'days_to_restrictions:Q',
            title='Policy Timing (days relative to first confirmed death)'
        ),
        y=alt.Y(
            'total_deaths_per_million:Q',
            title='COVID Deaths per Million'
        ),
        color=alt.Color(
            'continent:N',
            title='Continent'
        ),
        size=alt.Size(
            'population:Q',
            title='Population',
            scale=alt.Scale(range=[30, 2000])   #  bubble size
        ),
        tooltip=[
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('continent:N', title='Continent'),
            alt.Tooltip('days_to_restrictions:Q', title='Days to Restrictions'),
            alt.Tooltip('total_deaths_per_million:Q', title='Deaths per Million', format='.2f'),
            alt.Tooltip('population:Q', title='Population', format=',')
        ]
    )
)

# Regression line
regression = (
    alt.Chart(country_owid)
    .transform_regression(
        'days_to_restrictions',
        'total_deaths_per_million'
    )
    .mark_line(color='black', strokeDash=[6, 4], size=2)
    .encode(
        x='days_to_restrictions:Q',
        y='total_deaths_per_million:Q'
    )
)

# Labels for selected countries
labels_df = country_owid[country_owid['location'].isin(label_countries)]

labels = (
    alt.Chart(labels_df)
    .mark_text(
        align='left',
        baseline='middle',
        dx=7,
        dy=-7,
        fontSize=11
    )
    .encode(
        x='days_to_restrictions:Q',
        y='total_deaths_per_million:Q',
        text='location:N',
        color=alt.value('black')
    )
)

zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(
    color='gray',
    strokeDash=[4, 4]
).encode(
    x='x:Q'
)

# Combine 
chart = (
    (points + regression + labels + zero_line)
    .properties(
        width=800,
        height=500,
        title='Policy Timing and COVID Mortality by Country'
    )
    .configure_axis(
        labelFontSize=12,
        titleFontSize=13
    )
    .configure_legend(
        titleFontSize=12,
        labelFontSize=11
    )
)

chart

alt.LayerChart(...)

#### Visualization #4

In [9]:
owid = pd.read_csv('owid-covid-data.csv')

In [16]:
from vega_datasets import data as vega_data
import pycountry

owid['date'] = pd.to_datetime(owid['date'])

map_df = owid[
    owid['continent'].notna() &
    owid['iso_code'].notna() &
    owid['stringency_index'].notna()
].copy()
map_df = map_df[~map_df['iso_code'].str.startswith('OWID')]
map_df['month'] = map_df['date'].dt.to_period('M').dt.to_timestamp()

monthly_stringency = (
    map_df.groupby(['iso_code', 'location', 'month'], as_index=False)['stringency_index']
    .mean()
)

# Add numeric ISO id for topojson lookup
def alpha3_to_numeric(code):
    try:
        return int(pycountry.countries.get(alpha_3=code).numeric)
    except:
        return None

monthly_stringency['id'] = monthly_stringency['iso_code'].apply(alpha3_to_numeric)
monthly_stringency = monthly_stringency.dropna(subset=['id'])
monthly_stringency['id'] = monthly_stringency['id'].astype(int)
monthly_stringency['month_ts'] = monthly_stringency['month'].astype('int64') // 10**6

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

month_values = sorted(monthly_stringency['month'].unique())
monthly_stringency['month_idx'] = monthly_stringency['month'].map(
    {m: i for i, m in enumerate(month_values)}
)

slider = alt.binding_range(min=0, max=len(month_values) - 1, step=1, name='Month: ')
select_month = alt.param(name='selected_month', value=0, bind=slider)

map_chart = (
    alt.Chart(monthly_stringency)
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .transform_filter('datum.month_idx == selected_month')
    .transform_lookup(
        lookup='id',
        from_=alt.LookupData(
            alt.topo_feature(vega_data.world_110m.url, 'countries'),
            'id',
            ['geometry', 'type']
        )
    )
    .encode(
        color=alt.condition(
            'isValid(datum.stringency_index)',
            alt.Color('stringency_index:Q', title='Stringency Index',
                      scale=alt.Scale(scheme='orangered', domain=[0, 100])),
            alt.value('#ddd')
        ),
        tooltip=[
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('stringency_index:Q', title='Avg. Stringency', format='.1f')
        ]
    )
    .add_params(select_month)
    .properties(width=900, height=500, title='Monthly COVID Stringency Index by Country')
    .project('equalEarth')
)

map_chart

alt.Chart(...)

In [ ]:
# Prep data
owid['date'] = pd.to_datetime(owid['date'])

map_df = owid[
    owid['continent'].notna() &
    owid['iso_code'].notna() &
    owid['stringency_index'].notna()
].copy()

# Remove OWID aggregates like World, income groups, etc.
map_df = map_df[~map_df['iso_code'].str.startswith('OWID')]

# Convert to month
map_df['month'] = map_df['date'].dt.to_period('M').dt.to_timestamp()
map_df['month_label'] = map_df['month'].dt.strftime('%Y-%m')

# Monthly average stringency by country
monthly_stringency = (
    map_df.groupby(['iso_code', 'location', 'month_label'], as_index=False)['stringency_index']
    .mean()
)

# Plotly choropleth
fig = px.choropleth(
    monthly_stringency,
    locations='iso_code',
    color='stringency_index',
    hover_name='location',
    animation_frame='month_label',
    color_continuous_scale='OrRd',
    range_color=(0, 100),
    projection='natural earth',
    labels={'stringency_index': 'Stringency Index'},
    title='Monthly COVID Stringency Index by Country'
)

fig.update_layout(
    width=1000,
    height=600,
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

#### Visualization #4.2

In [ ]:

# -----------------------------
# 1) Choose countries
# -----------------------------
peer_countries = [
    "United States",
    "United Kingdom",
    "Canada",
    "Germany",
    "Italy",
    "Sweden",
    "Australia",
    "Japan"
]

# -----------------------------
# 2) Prep data
# -----------------------------
owid['date'] = pd.to_datetime(owid['date'])

ts_df = owid[
    owid['location'].isin(peer_countries)
][[
    'date',
    'location',
    'stringency_index',
    'new_deaths_per_million'
]].copy()

ts_df = ts_df.rename(columns={
    'new_deaths_per_million': 'mortality_rate'
})

# Drop rows where both metrics are missing
ts_df = ts_df[
    ts_df['stringency_index'].notna() | ts_df['mortality_rate'].notna()
]

# -----------------------------
# 3) Interactive country selection
# -----------------------------
selection = alt.selection_point(
    fields=['location'],
    bind='legend'
)

color = alt.Color(
    'location:N',
    title='Country'
)

# -----------------------------
# 4) Stringency chart
# -----------------------------
stringency_chart = (
    alt.Chart(ts_df)
    .mark_line(size=2)
    .encode(
        x=alt.X('date:T', title='Date'),
        y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
        color=color,
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('date:T', title='Date'),
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')
        ]
    )
    .add_params(selection)
    .properties(
        width=850,
        height=280,
        title='COVID Policy Stringency Over Time'
    )
)

# -----------------------------
# 5) Mortality chart
# -----------------------------
mortality_chart = (
    alt.Chart(ts_df)
    .mark_line(size=2)
    .encode(
        x=alt.X('date:T', title='Date'),
        y=alt.Y(
            'mortality_rate:Q',
            title='New Deaths per Million'
        ),
        color=color,
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('date:T', title='Date'),
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('mortality_rate:Q', title='New Deaths per Million', format='.2f')
        ]
    )
    .properties(
        width=850,
        height=280,
        title='COVID Mortality Trends Over Time'
    )
)

# -----------------------------
# 6) Combine charts
# -----------------------------
final_chart = alt.vconcat(
    stringency_chart,
    mortality_chart
).resolve_scale(
    color='shared'
).properties(
    title='Comparing COVID Policy and Mortality Trends Across Peer Countries'
)

final_chart

In [17]:

peer_countries = [
    "United States",
    "United Kingdom",
    "Canada",
    "Germany",
    "Italy",
    "Sweden",
    "Australia",
    "Japan"
]

owid['date'] = pd.to_datetime(owid['date'])

ts_df = owid[
    owid['location'].isin(peer_countries)
][[
    'date',
    'location',
    'stringency_index',
    'new_deaths_per_million'
]].copy()

ts_df = ts_df.rename(columns={
    'new_deaths_per_million': 'mortality_rate'
})

# Drop rows where both metrics are missing
ts_df = ts_df[
    ts_df['stringency_index'].notna() | ts_df['mortality_rate'].notna()
]

# Aggregate to week
ts_df['week'] = ts_df['date'].dt.to_period('W').dt.start_time

weekly_df = (
    ts_df.groupby(['week', 'location'], as_index=False)
    .agg({
        'stringency_index': 'mean',
        'mortality_rate': 'mean'
    })
)

selection = alt.selection_point(fields=['location'], bind='legend')

color = alt.Color('location:N', title='Country')

stringency_chart = (
    alt.Chart(weekly_df)
    .mark_line(size=2)
    .encode(
        x=alt.X('week:T', title='Date'),
        y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
        color=color,
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('week:T', title='Week'),
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')
        ]
    )
    .add_params(selection)
    .properties(width=850, height=280, title='COVID Policy Stringency Over Time')
)

mortality_chart = (
    alt.Chart(weekly_df)
    .mark_line(size=2)
    .encode(
        x=alt.X('week:T', title='Date'),
        y=alt.Y('mortality_rate:Q', title='New Deaths per Million'),
        color=color,
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip('week:T', title='Week'),
            alt.Tooltip('location:N', title='Country'),
            alt.Tooltip('mortality_rate:Q', title='New Deaths per Million', format='.2f')
        ]
    )
    .properties(width=850, height=280, title='COVID Mortality Trends Over Time')
)

final_chart = alt.vconcat(stringency_chart, mortality_chart).resolve_scale(
    color='shared'
).properties(
    title='Comparing COVID Policy and Mortality Trends Across Peer Countries'
)

final_chart

alt.VConcatChart(...)